In [3]:
import os
import pandas as pd
from pydub import AudioSegment

# --- Configuration ---
ROOT_DIR = '/Users/jakobmarrone/Documents/PlatformIO/Projects/LED BLINK/training/EmotionalCanines'
OUTPUT_DIR = "CreateML_Dataset"
TARGET_SR = 16000  # 16 kHz for Create ML
AUDIO_EXTENSION = ".wav" 

# Datasets mapped to their respective folders
DATASETS = [
    {"csv": "husky_train_labels.csv", "split": "Train", "audio_dir": os.path.join("husky", "train")},
    {"csv": "shiba_train_labels.csv", "split": "Train", "audio_dir": os.path.join("shiba", "train")},
    {"csv": "husky_test_labels.csv", "split": "Test", "audio_dir": os.path.join("husky", "test")},
    {"csv": "shiba_test_labels.csv", "split": "Test", "audio_dir": os.path.join("shiba", "test")}
]

def process_audio_data():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for ds in DATASETS:
        csv_path = os.path.join(ROOT_DIR, ds["csv"])
        source_audio_dir = os.path.join(ROOT_DIR, ds["audio_dir"])
        
        if not os.path.exists(csv_path):
            print(f"Skipping {ds['csv']}: File not found.")
            continue
            
        print(f"\nProcessing {ds['csv']}...")
        df = pd.read_csv(csv_path)
        
        for index, row in df.iterrows():
            audio_id = row['audio_id']
            arousal = row['arousal']
            valence = row['valence']
            
            # Combine arousal and valence for the Create ML class folder
            class_label = f"{arousal}_{valence}"
            
            source_file = os.path.join(source_audio_dir, f"{audio_id}{AUDIO_EXTENSION}")
            dest_folder = os.path.join(OUTPUT_DIR, ds["split"], class_label)
            os.makedirs(dest_folder, exist_ok=True)
            
            dest_file = os.path.join(dest_folder, f"{audio_id}.wav")
            
            if os.path.exists(source_file):
                try:
                    # pydub reads the file using FFmpeg, ignoring the incorrect extension
                    audio = AudioSegment.from_file(source_file)
                    
                    # Set frame rate to 16 kHz (16000)
                    audio = audio.set_frame_rate(TARGET_SR)
                    
                    # Convert to mono (Create ML prefers 1 channel for classification)
                    audio = audio.set_channels(1)
                    
                    # Export as a true, uncompressed WAV file
                    audio.export(dest_file, format="wav")
                    
                except Exception as e:
                    print(f"Error processing {source_file}: {repr(e)}")
            else:
                print(f"Warning: Audio file not found -> {source_file}")

    print(f"\n✅ Processing complete! Your data is ready in the '{OUTPUT_DIR}' folder.")

# if __name__ == "__main__":
#     process_audio_data()

In [4]:
process_audio_data()



Processing husky_train_labels.csv...

Processing shiba_train_labels.csv...

Processing husky_test_labels.csv...

Processing shiba_test_labels.csv...

✅ Processing complete! Your data is ready in the 'CreateML_Dataset' folder.
